In [1]:
!pip install pandas numpy scikit-learn catboost matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 22.8 MB/s eta 0:00:00


In [14]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

y = train["SalePrice"]
train.drop(["SalePrice"], axis=1, inplace=True)

# trainer ve testi birleştirme kodu
all_data = pd.concat([train, test], axis=0).reset_index(drop=True)

In [15]:
#ID kaldırma
all_data.drop("Id", axis=1, inplace=True)

In [16]:
#eksik verileri temizleme
#1-)yok anlamına gelenler
none_cols = [
    "Alley","PoolQC","Fence","MiscFeature","FireplaceQu",
    "GarageType","GarageFinish","GarageQual","GarageCond",
    "BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinType2"
]

for col in none_cols:
    all_data[col] = all_data[col].fillna("None")

#2-)sayısal eksikler 0a getirilir
num_cols = all_data.select_dtypes(include=["int64","float64"]).columns

for col in num_cols:
    all_data[col] = all_data[col].fillna(0)

#3-)Kategorisel eksikler none haline gelir
cat_cols = all_data.select_dtypes(include=["object"]).columns

for col in cat_cols:
    all_data[col] = all_data[col].fillna("None")

In [17]:
#Feature engineering
#1-)Mülk yaşı
all_data["HouseAge"] = 2010 - all_data["YearBuilt"]
all_data["RemodAge"] = 2010 - all_data["YearRemodAdd"]
all_data["GarageAge"] = 2010 - all_data["GarageYrBlt"]
#2-)Alan
all_data["TotalSF"] = (
    all_data["TotalBsmtSF"] +
    all_data["1stFlrSF"] +
    all_data["2ndFlrSF"]
)
#3-)Banyo sayısı
all_data["TotalBath"] = (
    all_data["FullBath"] +
    0.5 * all_data["HalfBath"] +
    all_data["BsmtFullBath"] +
    0.5 * all_data["BsmtHalfBath"]
)
#4-)bahçe dış alan sayısı
all_data["TotalPorchSF"] = (
    all_data["OpenPorchSF"] +
    all_data["EnclosedPorch"] +
    all_data["3SsnPorch"] +
    all_data["ScreenPorch"]
)

In [18]:
#Ordinal Encoding- Kategorik verileri derecelerine göre sayısal değerlere dönüştürme işlemidir.
#Model kategoriler arası ilişkiyi ve öncelik sırasını doğru şekilde anlar
qual_map = {
    "None":0, "Po":1, "Fa":2, "TA":3, "Gd":4, "Ex":5
}

qual_cols = [
    "ExterQual","ExterCond","BsmtQual","BsmtCond",
    "HeatingQC","KitchenQual","FireplaceQu",
    "GarageQual","GarageCond","PoolQC"
]

for col in qual_cols:
    all_data[col] = all_data[col].map(qual_map)
#None: Yok (bodrum şömine gibi)

#Po (Poor): Berbat

#Fa (Fair): Ortalama altı

#TA (Typical/Average): Ortalama

#Gd (Good): İyi

#Ex (Excellent): Mükemmel

In [19]:
#Kategorik Encode- modellerin metin tabanlı verileri işleyebilmesi için kategorik değişkenleri(kelimeleri) matematiksel ifadelere dönüştürme işlemi
#Transformerda daha farklı işlem olacak.
all_data = pd.get_dummies(all_data)

In [20]:
#Train ve testi ayırma
train_processed = all_data[:len(train)]
test_processed = all_data[len(train):]

In [21]:
#target(hedefin) dönüşümü
import numpy as np
y = np.log1p(y)